# Weighted OD Matrices by Mode — CAR / TRANSIT / RAIL / OTHER

Recreates the household-weighted Day 10 / Day 20 matrices from `THS_2018_MTX_weighted.ipynb`, split by an aggregated travel mode. `MODE_NAME` is mapped to four groups:

| Group | Modes |
|---|---|
| **CAR** | Vehicle as Driver, Vehicle as Passenger, Motorcycle/Moped |
| **TRANSIT** | Public Bus, Matronit, Special Taxi, Group Taxi |
| **RAIL** | Train |
| **OTHER** | Pedestrian, Default, Chartered Bus, Bicycle or other personal means, Other, Truck |

Trip extraction is unchanged (order by `INDIVID`/`tourID`/`ACT_ID`, origin = previous activity's `taz`, leaving time = `EndTime` for Home / `StartTime` otherwise, departures 6:00–9:00, each trip contributes its household's `wf_new`). **A trip's mode is the `MODE_NAME` of its destination activity row** — the mode used to arrive at the current activity.

In [1]:
import numpy as np
import pandas as pd

MODE_GROUP = {
    'Vehicle as Driver': 'CAR',
    'Pedestrian': 'OTHER',
    'Default': 'OTHER',
    'Public Bus': 'TRANSIT',
    'Vehicle as Passenger': 'CAR',
    'Matronit': 'TRANSIT',
    'Train': 'RAIL',
    'Special Taxi': 'TRANSIT',
    'Group Taxi': 'TRANSIT',
    'Chartered Bus': 'OTHER',
    'Motorcycle/Moped': 'CAR',
    'Bicycle or other personal means': 'OTHER',
    'Other': 'OTHER',
    'Truck': 'OTHER',
}
MODES = ['CAR', 'TRANSIT', 'RAIL', 'OTHER']

In [2]:
df = pd.read_csv('Input/Matrices/ACTIVITIES_DEC18_corrected.csv')
weights = pd.read_csv('Input/Matrices/households_with_weights.csv')[['HHID', 'wf_new']]
df = df.merge(weights, on='HHID', how='left', validate='many_to_one')
assert df['wf_new'].notna().all()

unmapped = set(df['MODE_NAME'].dropna().unique()) - set(MODE_GROUP)
assert not unmapped, f"MODE_NAME values missing from the dictionary: {unmapped}"
assert df['MODE_NAME'].notna().all()
df['mode_group'] = df['MODE_NAME'].map(MODE_GROUP)
print("mode-group distribution over all activity records:")
print(df['mode_group'].value_counts().to_string())

mode-group distribution over all activity records:
mode_group
CAR        88328
OTHER      73158
TRANSIT    10319
RAIL         724


In [3]:
def am_peak_trips(df_day):
    d = df_day.sort_values(by=['INDIVID', 'tourID', 'ACT_ID'])
    d['origin'] = d.groupby(['INDIVID', 'tourID'])['taz'].shift(1)
    d['destination'] = d['taz']
    d['StartTime'] = pd.to_datetime(d['StartTime'], dayfirst=True)
    d['EndTime'] = pd.to_datetime(d['EndTime'], dayfirst=True)
    d['leaving_time'] = pd.to_datetime(np.where(d['mainActivity'] == 'Home', d['EndTime'], d['StartTime']))
    mask = (d['leaving_time'].dt.hour >= 6) & (d['leaving_time'].dt.hour < 9)
    # a trip's mode = MODE_NAME of the destination (current) activity row
    f = d[mask].dropna(subset=['origin', 'destination'])
    # model-area trips only: both ends must have a real TAZ; Default arrival = no reported travel
    f = f[(f['origin'] != 0) & (f['destination'] != 0) & (f['MODE_NAME'] != 'Default')]
    return f

matrices = {}
summary_rows = []
all_mode_totals = {}
for day, df_day in [(10, df[df['ACT_DAY'] == 10]), (20, df[df['ACT_DAY'] == 20])]:
    trips = am_peak_trips(df_day.copy())
    all_mode_totals[day] = trips['wf_new'].sum()
    for mode in MODES:
        t = trips[trips['mode_group'] == mode]
        m = pd.crosstab(t['origin'], t['destination'], values=t['wf_new'], aggfunc='sum').fillna(0)
        matrices[(day, mode)] = m
        summary_rows.append({'day': day, 'mode': mode,
                             'sampled trips': len(t),
                             'expanded trips': t['wf_new'].sum()})

summary = pd.DataFrame(summary_rows).set_index(['day', 'mode'])
summary['share'] = summary['expanded trips'] / summary.groupby('day')['expanded trips'].transform('sum')
summary.round({'expanded trips': 0, 'share': 3})

sampled trips  expanded trips  share
day mode                                         
10  CAR               7642       1234371.0  0.563
    TRANSIT           1015        143215.0  0.065
    RAIL                24          3990.0  0.002
    OTHER             5739        811847.0  0.370
20  CAR               7492       1202092.0  0.556
    TRANSIT           1004        143776.0  0.066
    RAIL                33          4669.0  0.002
    OTHER             5668        812783.0  0.376

In [4]:
# consistency check: per day, the three mode matrices must sum to the all-mode weighted total
for day, ref_total in all_mode_totals.items():
    total = sum(matrices[(day, mode)].sum().sum() for mode in MODES)
    assert abs(total - ref_total) < 1.0, (day, total)
    print(f"day {day}: CAR + TRANSIT + RAIL + OTHER = {total:,.1f} expanded trips — matches the all-mode matrix")

day 10: CAR + TRANSIT + RAIL + OTHER = 2,193,422.5 expanded trips — matches the all-mode matrix
day 20: CAR + TRANSIT + RAIL + OTHER = 2,163,319.7 expanded trips — matches the all-mode matrix


In [5]:
import os
os.makedirs('Output', exist_ok=True)
for (day, mode), m in matrices.items():
    path = f'Output/matrix_{day}_weighted_{mode}.csv'
    m.to_csv(path)
    print(f"{path}:  shape {m.shape},  {m.sum().sum():,.0f} expanded trips")

Output/matrix_10_weighted_CAR.csv:  shape (617, 675),  1,234,371 expanded trips
Output/matrix_10_weighted_TRANSIT.csv:  shape (329, 332),  143,215 expanded trips
Output/matrix_10_weighted_RAIL.csv:  shape (11, 14),  3,990 expanded trips
Output/matrix_10_weighted_OTHER.csv:  shape (559, 606),  811,847 expanded trips


Output/matrix_20_weighted_CAR.csv:  shape (607, 664),  1,202,092 expanded trips
Output/matrix_20_weighted_TRANSIT.csv:  shape (333, 333),  143,776 expanded trips
Output/matrix_20_weighted_RAIL.csv:  shape (17, 15),  4,669 expanded trips
Output/matrix_20_weighted_OTHER.csv:  shape (572, 602),  812,783 expanded trips


## Notes

- The mode split applies to the *arriving* leg of each activity transition. `Default` (typically activities with no reported travel, e.g. the diary's opening record) falls under OTHER per the dictionary; such rows rarely survive the trip filter since a trip requires a previous activity in the same tour.
- Matrices only include origin/destination zones observed for that day-mode combination, so shapes differ across modes; align with `.reindex(...)` before comparing.
- The four matrices per day sum exactly to the corresponding all-mode weighted matrix (`matrix_10_weighted.csv` / `matrix_20_weighted.csv`).